# Harness de Evaluación — Extractor de Perfil de Estilo (M1)

Este notebook implementa un **harness ejecutable** que evalúa al modelo M1 (el extractor
fine-tuned con LoRA de `01_finetuning_lora_qwen25_oficial.ipynb`) sobre el **eval_set del
dominio** (`eval_set.json`), combinando **3 dimensiones** de evaluación:

- **Dimensión 1 — Métricas automáticas clásicas:** como esto es un problema de
  clasificación multi-campo, medimos exact-match y F1 macro por campo (con la misma
  lógica del notebook de entrenamiento).

- **Dimensión 2 — LLM como juez:** un segundo modelo Qwen (`Qwen/Qwen2.5-1.5B-Instruct`
  o uno menor) evalúa cada respuesta con una **rúbrica explícita de 1 a 5** que incluye
  ejemplos ancla.

- **Dimensión 3 — Cumplimiento de los criterios del campo:** mide cuántas respuestas
  cumplen el `criterion` declarado en cada caso del eval_set (p.ej. juez >= 4, o la
  verificación específica de JSON vacío para las trampas out-of-domain).

El punto de entrada principal es la función `harness(eval_set, system, judge_backend)`
que ejecuta las 3 dimensiones y devuelve un **scorecard** consolidado.

> **Entorno:** pensado para Colab con GPU T4.
> Si no hay GPU/dependencias, el notebook cae a un _modo demo_ determinista para que el
> flujo completo sea ejecutable y testeable.

## 0. Setup

In [ ]:
!pip install -q -U transformers==4.46.2 accelerate==1.1.1 peft==0.13.2 \
    datasets==3.1.0 scikit-learn==1.5.2

In [ ]:
import json
import os
import random
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

In [ ]:
# ============================================================
# Configuración de Google Drive (Colab)
# ============================================================
# En Colab montamos Drive y todo se lee/guarda en la MISMA carpeta que usa el
# notebook de entrenamiento (01...): MiUnidad/personal-shopper-ia
#   - eval_set.json               -> BASE_DIR/eval_set.json
#   - adaptador LoRA (M1)         -> BASE_DIR/qwen25-1.5b-style-extractor-lora/adapter_final
# En local (sin Colab) BASE_DIR queda en '.' y se usan los archivos del repo.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    EN_COLAB = True
except Exception:
    EN_COLAB = False
    print("No estamos en Colab: se usarán los archivos locales del proyecto.")

BASE_DIR = "/content/drive/MyDrive/personal-shopper-ia" if EN_COLAB else "."
EVAL_SET_PATH = f"{BASE_DIR}/eval_set.json"
ADAPTER_PATH = f"{BASE_DIR}/qwen25-1.5b-style-extractor-lora/adapter_final"
print("BASE_DIR   :", BASE_DIR)
print("Eval set   :", EVAL_SET_PATH)
print("LoRA (M1)  :", ADAPTER_PATH)


---

## 1. Ontología y prompt del sistema

Definimos la **ontología** (los 5 campos con sus valores permitidos) y el **prompt de
sistema** del extractor. Son idénticos a los del notebook de entrenamiento M1, para que
la comparación entre baseline y fine-tuned sea honesta y la evaluación use exactamente
el mismo espacio de salida.

In [ ]:
ONTOLOGIA: Dict[str, List[str]] = {
    "estilo": ["Casual", "Formal", "Minimalista", "Urbano", "Bohemio", "Deportivo", "Clasico"],
    "ocasion": ["boda", "trabajo", "fin_de_semana", "viaje", "deporte", "evento_formal"],
    "clima": ["calido", "frio", "templado"],
    "paleta": ["neutros", "pasteles", "oscuros", "colores_vivos", "monocromatico"],
    "fit": ["holgado", "regular", "ajustado", "oversized"],
}
CAMPOS = list(ONTOLOGIA.keys())

SYSTEM_PROMPT = (
    "Eres un asistente que extrae el perfil de estilo de un cliente a partir de su mensaje "
    "de chat en una tienda de moda online.\n"
    "Debes devolver UNICAMENTE un objeto JSON con los campos que el cliente haya mencionado "
    "o se puedan inferir claramente, usando SOLO estos campos y valores posibles:\n\n"
    "- estilo: Casual, Formal, Minimalista, Urbano, Bohemio, Deportivo, Clasico\n"
    "- ocasion: boda, trabajo, fin_de_semana, viaje, deporte, evento_formal\n"
    "- clima: calido, frio, templado\n"
    "- paleta: neutros, pasteles, oscuros, colores_vivos, monocromatico\n"
    "- fit: holgado, regular, ajustado, oversized\n\n"
    "No incluyas campos que no se puedan inferir del mensaje. No agregues texto fuera del "
    "JSON. No inventes valores fuera de esta lista."
)

print("Ontología:", ONTOLOGIA)

---

## 2. Carga del eval_set del dominio

Cargamos `eval_set.json`: el conjunto de evaluación _de campo_ con ejemplos gold. Cada
ejemplo tiene `id`, `tipo` (estándar / ambigüo / trampa out-of-domain), `input`,
`expected_response` y `criterion`. Esta es la base sobre la que se calculan las tres
dimensiones.

In [ ]:
def load_evalset(path: Optional[Union[str, Path]] = None) -> List[Dict[str, Any]]:
    """Carga el eval_set del dominio (lista de ejemplos input/expected/criterion).

    Prioridad de búsqueda:
      1. la ruta explícita `path` si se pasa;
      2. el eval_set en BASE_DIR (Google Drive en Colab);
      3. eval_set.json en el directorio local.
    """
    candidatas = []
    if path:
        candidatas.append(str(path))
    candidatas.append(EVAL_SET_PATH)
    candidatas.append("eval_set.json")
    for c in candidatas:
        if Path(c).exists():
            with open(c, "r", encoding="utf-8") as f:
                data = json.load(f)
            assert isinstance(data, list) and len(data) >= 1, \
                "eval_set debe ser una lista no vacía"
            print("Eval set cargado desde:", c)
            return data
    raise FileNotFoundError(f"No se encontró eval_set.json. Buscado en: {candidatas}")


EVAL_SET = load_evalset()
print(f"Total de casos: {len(EVAL_SET)}")
from collections import Counter
print("Distribución por tipo:", dict(Counter(e.get('tipo', 'estandar') for e in EVAL_SET)))


---

## 3. Dimensión 1 — Métricas automáticas clásicas (clasificación)

El extractor produce un JSON sobre un espacio de clasificación multi-campo. 

1. **`extraer_json`**: aísla el primer objeto JSON del texto generado y lo valida contra la
   ontología (descarta campos/valores fuera de la lista → anti-alucinación por formato).
2. **JSON válido**: fracción de respuestas que produjeron un JSON parseable y válido.
3. **Exact-match**: fracción de respuestas cuyo JSON coincide **exactamente** con el gold
   (los 5 campos correctos a la vez y sin extras inventados).
4. **F1 macro por campo**: métrica de clasificación multi-clase por campo (con sentinela
   `__NA__` para "no lo predijo / JSON inválido").

In [ ]:
def extraer_json(texto: str) -> Optional[Dict[str, Any]]:
    """Extrae y valida el primer objeto JSON del texto contra la ontología.
    Devuelve None si no hay JSON parseable. Filtra campos/valores fuera de la ontología."""
    if not texto:
        return None
    if isinstance(texto, dict):
        obj = texto
    else:
        try:
            inicio = texto.find("{")
            fin = texto.rfind("}")
            if inicio == -1 or fin == -1 or fin < inicio:
                return None
            obj = json.loads(texto[inicio:fin + 1])
        except (json.JSONDecodeError, TypeError):
            return None
    if not isinstance(obj, dict):
        return None
    return {k: v for k, v in obj.items() if k in ONTOLOGIA and v in ONTOLOGIA[k]}


def f1_macro_por_campo(predicciones: List[Dict], verdaderos: List[Dict]) -> Dict[str, float]:
    """F1 macro por campo donde el campo SÍ aplica en el ground truth.
    Sentinela '__NA__' para valores no predichos / JSON inválido."""
    try:
        from sklearn.metrics import f1_score
    except ImportError:
        f1_score = None

    f1_por_campo: Dict[str, float] = {}
    for campo in CAMPOS:
        y_true, y_pred = [], []
        for pred, verdad in zip(predicciones, verdaderos):
            if campo not in verdad:
                continue  # solo evaluamos donde el campo SÍ aplica en el ground truth
            y_true.append(verdad[campo])
            valor = pred.get(campo) if pred else None
            y_pred.append(valor if valor in ONTOLOGIA[campo] else "__NA__")
        if not y_true:
            continue
        etiquetas = ONTOLOGIA[campo] + ["__NA__"]
        if f1_score is not None:
            f1_por_campo[campo] = float(f1_score(y_true, y_pred, labels=etiquetas, average="macro", zero_division=0))
        else:
            punt = 0.0
            for lab in etiquetas:
                tp = sum(1 for a, b in zip(y_true, y_pred) if a == b == lab)
                fp = sum(1 for a, b in zip(y_true, y_pred) if a != lab and b == lab)
                fn = sum(1 for a, b in zip(y_true, y_pred) if a == lab and b != lab)
                prec = tp / (tp + fp) if (tp + fp) else 0.0
                rec = tp / (tp + fn) if (tp + fn) else 0.0
                f = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
                punt += f
            f1_por_campo[campo] = punt / len(etiquetas)
    return f1_por_campo


def dimension_1(textos_generados: List[str], salidas_verdaderas: List[Dict]) -> Dict[str, Any]:
    """Dimensión 1: tasa JSON válido + exact-match + F1 macro por campo."""
    n = max(1, len(textos_generados))
    predicciones = [extraer_json(t) for t in textos_generados]
    json_valido = sum(1 for p in predicciones if p is not None) / n
    exact_match = sum(1 for p, v in zip(predicciones, salidas_verdaderas)
                      if p is not None and p == v) / n
    f1_por_campo = f1_macro_por_campo(predicciones, salidas_verdaderas)
    f1_promedio = float(sum(f1_por_campo.values()) / len(f1_por_campo)) if f1_por_campo else 0.0
    return {
        "json_valido": json_valido,
        "exact_match": exact_match,
        "f1_por_campo": f1_por_campo,
        "f1_promedio": f1_promedio,
        "predicciones": predicciones,
    }

---

## 4. Dimensión 2 — LLM como juez (rúbrica 1-5 con ejemplos ancla)

Aquí un **segundo modelo Qwen** (por defecto `Qwen/Qwen2.5-1.5B-Instruct`, configurable a
uno menor) actúa como juez. Se le da una **rúbrica explícita de 1 a 5** con **ejemplos
ancla** para cada puntaje, más el mensaje del cliente, el gold y la salida del sistema.
El juez responde SOLO un JSON `{"puntaje": 1-5, "justificacion": "..."}`.

La rúbrica penaliza específicamente la **alucinación** (inventar campos como `ocasion` o
`paleta` que no se mencionaron) y contempla los casos ambigüos/out-of-domain (donde un
gold `{}` debe recibir 5 si el sistema no inventa nada).

> El backend del juez es **local (Qwen)**. También existe un `mini` (juez determinista
> offline) como respaldo para ejecutar el flujo sin GPU.

In [ ]:
RUBRICA = """
Eres un juez experto evaluando a un modelo de extracción de perfil de estilo de moda.
Recibes el mensaje del cliente, la salida JSON esperada (gold) y la salida JSON del sistema.
Evalúa la CALIDAD de la salida del sistema con un puntaje del 1 al 5 según esta rúbrica:

Rúbrica (1-5):
- 5 (Perfecto): El JSON coincide exactamente con el gold. Solo incluye los campos
  mencionados/inferibles, con valores válidos de la ontología y sin alucinaciones.
- 4 (Bueno): Todos los campos clave están correctos y no hay campos inventados graves,
  pero difiere en un matiz menor (p.ej. 'holgado' vs 'oversized') o un campo secundario.
- 3 (Regular): Acierta los campos principales pero omite algún campo clave o introduce
  una alucinación leve (p.ej. un campo no mencionado que no rompe la intención).
- 2 (Malo): Errores importantes: campos clave equivocados u omisión de varios campos.
- 1 (Muy malo): JSON inválido, valores fuera de la ontología, o alucina campos/valores
  inventados que contradicen el mensaje (p.ej. inventar 'ocasion' o 'paleta' ausentes).

Ejemplos ancla:
- Ancla 5: input="quiero algo urbano, oversized, en negro" -> gold={"estilo":"Urbano","fit":"oversized","paleta":"oscuros"}
           salida={"estilo":"Urbano","fit":"oversized","paleta":"oscuros"} -> 5 (coincidencia exacta, sin extras).
- Ancla 4: el mismo input, salida={"estilo":"Urbano","fit":"holgado","paleta":"oscuros"} -> 4 (solo matiz oversize/holgado).
- Ancla 3: el mismo input, salida={"estilo":"Urbano","fit":"oversized"} -> 3 (omite paleta).
- Ancla 2: el mismo input, salida={"estilo":"Formal","fit":"oversized","paleta":"oscuros"} -> 2 (estilo clave errado).
- Ancla 1: el mismo input, salida={"estilo":"Urbano","fit":"oversized","paleta":"monocromatico","ocasion":"viaje"} -> 1 (alucina ocasion).

NOTA IMPORTANTE para casos ambigüos u out-of-domain: si el gold es {} (consulta fuera
del alcance), una salida {} o un aviso de fuera de alcance es 5; inventar cualquier campo
es como máximo 2 (alucinación).

Responde SOLO con un JSON: {"puntaje": <1-5>, "justificacion": "<1-2 frases>"}
"""

### 4.1 Backends del juez: local (Qwen) y mini (determinista)

In [ ]:
def _parse_juicio(texto: str) -> Dict[str, Any]:
    """Extrae {'puntaje', 'justificacion'} del output del juez (tolera ruido)."""
    m = re.search(r'"puntaje"\s*:\s*(\d+)', texto)
    if m:
        puntaje = int(m.group(1))
        just = re.search(r'"justificacion"\s*:\s*"([^"]*)"', texto)
        return {"puntaje": puntaje, "justificacion": just.group(1) if just else ""}
    for p in range(5, 0, -1):
        if re.search(rf"\b{p}\b", texto):
            return {"puntaje": p, "justificacion": texto.strip()[:200]}
    return {"puntaje": 1, "justificacion": "No se pudo parsear la rúbrica del juez."}


def _juez_minirubrica(texto_input: str, gold: Dict, salida: Dict) -> Dict[str, Any]:
    """Mini-juez determinista offline: compara contra el gold (respaldo sin LLM)."""
    if isinstance(salida, dict) and isinstance(gold, dict):
        if salida == gold:
            return {"puntaje": 5, "justificacion": "Coincidencia exacta con el gold (minijuez offline)."}
        intentos = 0
        aciertos = 0
        for k, v in gold.items():
            intentos += 1
            aciertos += 1 if salida.get(k) == v else 0
        return {
            "puntaje": int(round(1 + 4 * (aciertos / max(1, intentos)))),
            "justificacion": f"Minijuez offline: {aciertos}/{intentos} campos correctos vs gold.",
        }
    return {"puntaje": 1, "justificacion": "JSON inválido o no comparable (minijuez offline)."}


def _juez_local(input_texto: str, gold: Dict, salida: Dict, cfg: Dict) -> Dict[str, Any]:
    """Juez LLM usando un modelo Qwen local (M1 reutilizado o Qwen/Qwen2.5-0.5B-Instruct).
    El modelo del juez vive en `cfg['judge_model']` (un system con `.model_chat`)."""
    judge_system = cfg.get("judge_model")
    # Solo usamos el LLM si es un modelo REAL (tiene .model cargado). En modo demo (sin
    # modelo) caemos al minijuez determinista para que el flujo siga siendo ejecutable.
    if judge_system is None or judge_system.get("model") is None:
        return _juez_minirubrica(input_texto, gold, salida)
    user_payload = (
        f"MENSAJE DEL CLIENTE:\n{input_texto}\n\n"
        f"GOLD (esperado):\n{json.dumps(gold, ensure_ascii=False)}\n\n"
        f"SALIDA DEL SISTEMA:\n{json.dumps(salida, ensure_ascii=False)}"
    )
    mensajes = [
        {"role": "system", "content": RUBRICA},
        {"role": "user", "content": user_payload},
    ]
    texto = judge_system["model_chat"](mensajes, max_new_tokens=200)
    juicio = _parse_juicio(texto)
    # Si el LLM no devolvió un puntaje válido 1-5, caemos al minijuez (robustez).
    if not (1 <= juicio["puntaje"] <= 5):
        return _juez_minirubrica(input_texto, gold, salida)
    return juicio


def dimension_2(eval_set: List[Dict], predicciones: List[Dict],
                judge_backend: Dict[str, Any]) -> Dict[str, Any]:
    """Dimensión 2: LLM como juez con rúbrica 1-5. Devuelve puntajes por caso + promedio."""
    kind = judge_backend.get("kind", "mini")
    juicios = []
    for i, ejemplo in enumerate(eval_set):
        gold = ejemplo["expected_response"]
        salida = predicciones[i] if i < len(predicciones) else None
        if kind == "local":
            juicio = _juez_local(ejemplo["input"], gold, salida or {}, judge_backend)
        else:
            juicio = _juez_minirubrica(ejemplo["input"], gold, salida or {})
        juicios.append({"id": ejemplo["id"], **juicio})
    puntajes = [j["puntaje"] for j in juicios]
    return {
        "juicios": juicios,
        "puntaje_promedio": float(sum(puntajes) / len(puntajes)) if puntajes else 0.0,
        "distribucion": {p: puntajes.count(p) for p in range(1, 6)},
    }

---

## 5. Dimensión 3 — Cumplimiento de los criterios del campo

Esta dimensión mide **cuántas respuestas cumplen los criterios declarados en el eval_set**
de campo. Regla de cumplimiento por caso:

- Para casos **estándar / ambigüos**: el criterio es `puntaje del juez >= umbral`
  (por defecto 4).
- Para **trampas out-of-domain** (cuyo gold es `{}`): hay una **verificación específica**
  del dominio — la respuesta debe ser un JSON vacío (o `None`), porque inventar cualquier
  campo (p.ej. un `fit` por "talla 32") es alucinación. Esta verificación es independiente
  del juez: un modelo que alucina falla D3 aunque el juez puntúe alto.

El resultado es una **fracción de cumplimiento** = cumplidos / total.

In [ ]:
def dimension_3(eval_set: List[Dict], juicios: List[Dict], predicciones: List[Dict],
                umbral_judge: int = 4) -> Dict[str, Any]:
    """Dimensión 3: ¿cuántas respuestas cumplen los criterios declarados en el eval_set?

    - Casos con gold {} (out-of-domain): exigen JSON vacío (verificación específica).
    - Resto: criterio = judge >= umbral_judge (default 4)."""
    cumplidos = 0
    detalles = []
    for i, ejemplo in enumerate(eval_set):
        gold = ejemplo["expected_response"]
        salida = predicciones[i] if i < len(predicciones) else None
        juicio = juicios[i] if i < len(juicios) else {"puntaje": 0}

        es_trampa = gold == {}
        if es_trampa:
            # verificación específica del dominio: no debe inventar campos
            cumple = (salida == {} or salida is None or salida is False)
        else:
            cumple = juicio["puntaje"] >= umbral_judge

        detalles.append({
            "id": ejemplo["id"],
            "tipo": ejemplo.get("tipo", "estandar"),
            "cumple": bool(cumple),
            "juicio": juicio["puntaje"],
            "salida": salida,
            "gold": gold,
        })
        cumplidos += int(cumple)

    total = max(1, len(eval_set))
    return {
        "cumplimiento": cumplidos / total,
        "cumplidos": cumplidos,
        "total": total,
        "umbral_judge": umbral_judge,
        "detalles": detalles,
    }

---

## 6. Carga del sistema M1 (extractor) y del juez Qwen

Definimos un **cargador genérico de modelos Qwen** (`make_qwen_system`) que crea un objeto
`system` con `.generate`/`.model_chat`. Se usa para:

- **M1 (extractor)**: `make_system(adapter_dir)` → Qwen base + adaptador LoRA
  (`qwen25-1.5b-style-extractor-lora/adapter_final`).
- **Juez (Dimensión 2)**: `make_judge_model()` → `Qwen/Qwen2.5-1.5B-Instruct` o un modelo
  menor (configurable vía `JUDGE_MODEL`), cargado SIN adaptador.

Si no hay GPU/dependencias, se cae a un **modo demo** determinista para que el notebook sea
ejecutable de punta a punta.

In [ ]:
def make_qwen_system(model_dir: Optional[str] = None,
                     base_model: str = "Qwen/Qwen2.5-1.5B-Instruct",
                     device: str = "auto", max_new_tokens: int = 128) -> Dict[str, Any]:
    """Carga un modelo Qwen (opcionalmente con LoRA) y devuelve un system con .generate
    y .model_chat. Si no hay deps/GPU, devuelve un system demo determinista.

    IMPORTANTE para cargar la LoRA: el adaptador guardado contiene SOLO los pesos de bajo
    rango (aprox. 4.3M parámetros), NO el modelo base completo. Por eso:
        1. Cargar el modelo base Qwen con AutoModelForCausalLM (base_model).
        2. Aplicar el adaptador encima con PeftModel.from_pretrained(base, adapter_dir).
    El objeto devuelto expone `lora_cargada` para VERIFICAR que la LoRA se aplicó.
    """
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        device_map = device if device != "auto" else "auto"

        # El tokenizador y el modelo SIEMPRE se cargan desde el modelo base (Qwen).
        tokenizer = AutoTokenizer.from_pretrained(base_model)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            base_model,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map=device_map,
        )

        lora_cargada = False
        if model_dir:
            try:
                from peft import PeftModel
                model = PeftModel.from_pretrained(model, model_dir)
                lora_cargada = True
                print(f"[harness] LoRA APLICADA correctamente sobre {base_model}.")
            except Exception as exc:
                print(f"[harness] No se pudo cargar la LoRA ({exc}). Usando modelo base SIN LoRA.")

        if hasattr(model, "config") and hasattr(model.config, "use_cache"):
            model.config.use_cache = True

        def model_chat(mensajes, max_new_tokens=max_new_tokens):
            texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(texto, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=max_new_tokens,
                    pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                )
            return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        def generate(mensaje_user: str, system_prompt: str = SYSTEM_PROMPT) -> str:
            return model_chat([
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": mensaje_user},
            ])

        dev = next(model.parameters()).device
        print(f"[harness] Qwen cargado: {base_model} en {dev} (LoRA={'si' if lora_cargada else 'no'})")
        return {"model": model, "tokenizer": tokenizer, "generate": generate, "model_chat": model_chat,
                "modo": "real", "lora_cargada": lora_cargada, "base_model": base_model}

    except Exception as exc:
        print(f"[harness] Modo DEMO (sin deps/GPU): {exc}")
        return make_system_demo(max_new_tokens=max_new_tokens)


def _buscar_adaptador() -> Optional[str]:
    """Busca el adaptador LoRA: primero en BASE_DIR (Drive en Colab), luego local."""
    candidatas = [
        ADAPTER_PATH,
        f"{BASE_DIR}/qwen25-1.5b-style-extractor-lora/adapter_final",
        "/content/qwen25-1.5b-style-extractor-lora/adapter_final",
        "qwen25-1.5b-style-extractor-lora/adapter_final",
    ]
    for c in candidatas:
        if Path(c).exists():
            return c
    return None


def make_system(adapter_dir: Optional[str] = None,
                base_model: str = "Qwen/Qwen2.5-1.5B-Instruct", device: str = "auto",
                max_new_tokens: int = 128) -> Dict[str, Any]:
    """Carga el extractor M1 (Qwen + adaptador LoRA) como sistema principal.
    Si no encuentra el adaptador, levanta el modelo base igual (sin LoRA) pero lo avisa."""
    if adapter_dir is None:
        adapter_dir = _buscar_adaptador()
    if adapter_dir and not Path(adapter_dir).exists():
        print(f"[harness] No existe el adaptador {adapter_dir}; usando {base_model} sin LoRA.")
        adapter_dir = None
    return make_qwen_system(adapter_dir, base_model, device, max_new_tokens)


def make_judge_model(model_id: Optional[str] = None,
                     device: str = "auto", max_new_tokens: int = 200) -> Dict[str, Any]:
    """Carga el modelo juez (Dimensión 2).
    Por defecto usa Qwen/Qwen2.5-0.5B-Instruct, MUCHO más ligero para caber en free Colab
    junto al extractor M1. Configura con JUDGE_MODEL=... si quieres otro."""
    judge_id = model_id or os.environ.get("JUDGE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
    sys_model = make_qwen_system(None, judge_id, device, max_new_tokens)
    sys_model["modo"] = "juez"
    return sys_model

In [ ]:
def make_system_demo(max_new_tokens: int = 128) -> Dict[str, Any]:
    """System determinista de demostración: imita al extractor para tests del harness."""
    def generate(mensaje_user: str, system_prompt: str = SYSTEM_PROMPT) -> str:
        texto = mensaje_user.lower()
        salida: Dict[str, Any] = {}
        pares = {
            "boda": ("ocasion", "boda"), "casamiento": ("ocasion", "boda"),
            "trabaj": ("ocasion", "trabajo"), "oficina": ("ocasion", "trabajo"),
            "viaje": ("ocasion", "viaje"), "fin de semana": ("ocasion", "fin_de_semana"),
            "finde": ("ocasion", "fin_de_semana"), "gimnasio": ("ocasion", "deporte"),
            "trotar": ("ocasion", "deporte"), "ejercicio": ("ocasion", "deporte"),
            "gala": ("ocasion", "evento_formal"), "formal": ("ocasion", "evento_formal"),
            "frio": ("clima", "frio"), "abrigad": ("clima", "frio"),
            "caliente": ("clima", "calido"), "calor": ("clima", "calido"),
            "templado": ("clima", "templado"), "urbano": ("estilo", "Urbano"),
            "urbana": ("estilo", "Urbano"), "minimal": ("estilo", "Minimalista"),
            "bohem": ("estilo", "Bohemio"), "clasico": ("estilo", "Clasico"),
            "deportiv": ("estilo", "Deportivo"), "casual": ("estilo", "Casual"),
            "negro": ("paleta", "oscuros"), "oscuro": ("paleta", "oscuros"),
            "pastel": ("paleta", "pasteles"), "neutro": ("paleta", "neutros"),
            "vivos": ("paleta", "colores_vivos"), "un solo color": ("paleta", "monocromatico"),
            "monocrom": ("paleta", "monocromatico"),
            "oversized": ("fit", "oversized"), "ancha": ("fit", "oversized"),
            "holgad": ("fit", "holgado"), "regular": ("fit", "regular"),
            "ajustad": ("fit", "ajustado"), "al cuerpo": ("fit", "ajustado"),
            "pega": ("fit", "ajustado"),
        }
        for clave, (campo, valor) in pares.items():
            if clave in texto and campo not in salida:
                salida[campo] = valor
        # trampas out-of-domain: envíos/tallas/presupuesto -> JSON vacío
        if any(k in texto for k in ["envio", "descuento", "presupuesto", "talla "]):
            salida = {}
        return json.dumps(salida, ensure_ascii=False)

    def model_chat(mensajes, max_new_tokens=max_new_tokens):
        user = next((m["content"] for m in mensajes if m["role"] == "user"), "")
        return generate(user)

    return {"model": None, "tokenizer": None, "generate": generate, "model_chat": model_chat, "modo": "demo"}

---

## 7. Función principal: `harness(eval_set, system, judge_backend)`

Combina las **3 dimensiones** sobre el sistema dado y devuelve un **scorecard**
consolidado. El flujo es:

1. Ejecutar el sistema sobre cada `input` del eval_set.
2. **D1**: métricas clásicas (exact-match + F1 macro por campo + JSON válido).
3. **D2**: evaluar cada respuesta con el LLM juez (rúbrica 1-5).
4. **D3**: fracción de respuestas que cumplen los criterios del campo.

`judge_backend` acepta `{"kind": "mini"}` (determinista, sin GPU) o
`{"kind": "local", "judge_model": <system Qwen>}` (Qwen local).

In [ ]:
def harness(eval_set: Union[List[Dict], str, Path], system: Dict[str, Any],
            judge_backend: Union[str, Dict[str, Any]] = "mini",
            prompt: str = SYSTEM_PROMPT, umbral_judge: int = 4) -> Dict[str, Any]:
    """Ejecuta las 3 dimensiones de evaluación sobre el sistema y devuelve el scorecard.

    - eval_set: lista de ejemplos, o ruta a eval_set.json.
    - system: objeto devuelto por `make_system(...)` (expone `.generate`).
    - judge_backend: "mini" (determinista) o {"kind": "local", "judge_model": <Qwen>}.
    - prompt: prompt de sistema para el extractor.
    - umbral_judge: umbral (>=) para considerar cumplido el criterio del campo.
    """
    if isinstance(eval_set, (str, Path)):
        eval_set = load_evalset(str(eval_set))

    if isinstance(judge_backend, str):
        judge_backend = {"kind": judge_backend}

    # ---- ejecutar el sistema sobre todos los inputs ----
    textos_generados = [
        system["generate"](ejemplo["input"], system_prompt=prompt)
        for ejemplo in eval_set
    ]
    verdaderos = [ejemplo["expected_response"] for ejemplo in eval_set]

    # ---- Dimensión 1 ----
    d1 = dimension_1(textos_generados, verdaderos)

    # ---- Dimensión 2 ----
    d2 = dimension_2(eval_set, d1["predicciones"], judge_backend)

    # ---- Dimensión 3 ----
    d3 = dimension_3(eval_set, d2["juicios"], d1["predicciones"], umbral_judge=umbral_judge)

    # ---- scorecard consolidado ----
    scorecard = {
        "modelo": system.get("modo", "real"),
        "n_ejemplos": len(eval_set),
        "d1_metricas_clasicas": d1,
        "d2_llm_judge": d2,
        "d3_cumplimiento_dominio": d3,
        "resumen": {
            "D1_exact_match": d1["exact_match"],
            "D1_f1_macro_promedio": d1["f1_promedio"],
            "D1_json_valido": d1["json_valido"],
            "D2_puntaje_judge_promedio": d2["puntaje_promedio"],
            "D3_cumplimiento_criterios_campo": d3["cumplimiento"],
        },
        "config": {"judge_backend": judge_backend.get("kind"), "umbral_judge": umbral_judge},
    }
    return scorecard


def imprimir_scorecard(scorecard: Dict[str, Any]) -> None:
    print("=" * 62)
    print(f"SCORECARD — Modelo: {scorecard['modelo']} | Casos: {scorecard['n_ejemplos']}")
    print("=" * 62)
    r = scorecard["resumen"]
    print("\n[D1] Métricas clásicas de clasificación")
    print(f"     JSON válido       : {r['D1_json_valido']:.1%}")
    print(f"     Exact-match       : {r['D1_exact_match']:.1%}")
    print(f"     F1 macro promedio : {r['D1_f1_macro_promedio']:.3f}")
    print("     F1 por campo      : " + ", ".join(
        f"{k}={v:.3f}" for k, v in scorecard["d1_metricas_clasicas"]["f1_por_campo"].items()))
    print(f"\n[D2] LLM como juez (rúbrica 1-5, backend={scorecard['config']['judge_backend']})")
    print(f"     Puntaje promedio  : {r['D2_puntaje_judge_promedio']:.2f} / 5")
    print("     Distribución      : " + ", ".join(
        f"{k}★={v}" for k, v in scorecard["d2_llm_judge"]["distribucion"].items()))
    print(f"\n[D3] Cumplimiento de criterios del campo (judge>={scorecard['config']['umbral_judge']})")
    print(f"     Cumplimiento      : {r['D3_cumplimiento_criterios_campo']:.1%} "
          f"({scorecard['d3_cumplimiento_dominio']['cumplidos']}/{scorecard['d3_cumplimiento_dominio']['total']})")
    print("\nDetalle por caso (D3):")
    for det in scorecard["d3_cumplimiento_dominio"]["detalles"]:
        print(f"     [{'OK ' if det['cumple'] else 'FAIL'}] {det['id']:9s} "
              f"tipo={det['tipo']:18s} judge={det['juicio']}/5")

## 8. Ejecución del harness sobre M1

### 8.1 Cargar el modelo M1 (extractor)

Carga el adaptador LoRA que el notebook de entrenamiento guardó en **Google Drive**
(`BASE_DIR/qwen25-1.5b-style-extractor-lora/adapter_final`). Si no existe / no hay GPU,
cae a modo demo (heurística determinista) para validar el flujo.


In [ ]:
# En Colab el adaptador se entrena y guarda en DRIVE desde el notebook 01
# (BASE_DIR/qwen25-1.5b-style-extractor-lora/adapter_final). Lo cargamos desde ahí.
ADAPTER_DIR = ADAPTER_PATH
system = make_system(ADAPTER_DIR)
print("LoRA cargada en el extractor:", system.get("lora_cargada", False))


## 8.2 Elegir y cargar el juez (Dimensión 2) — sin reventar la RAM

El juez es un **Qwen local**. Para que corra en **free Colab (T4, ~12 GB de RAM)**, la clave
es **no cargar un segundo modelo completo**. En la celda siguiente puedes elegir:

- **`USAR_MISMO_MODELO_JUEZ = True` (recomendado):** el M1 ya cargado hace doble función de
  extractor y juez. Costo extra de RAM ≈ 0, porque no se levanta ningún modelo nuevo.

- **`USAR_MISMO_MODELO_JUEZ = False` + `JUEZ_LOCAL_APARTE = True`:** carga un juez Qwen
  **menor** (`Qwen/Qwen2.5-0.5B-Instruct`, configurable con `JUDGE_MODEL`) separado.
  Más RAM que la opción anterior, pero mucho más ligero que un segundo 1.5B.

- **`USAR_MISMO_MODELO_JUEZ = False` + `JUEZ_LOCAL_APARTE = False`:** **minijuez determinista**
  offline (sin modelo), solo para validar el flujo del notebook sin GPU.

> El error de RAM típico aparece al ~8GB de los ~12GB disponibles: casi siempre es porque
> se estaba cargando un SEGUNDO Qwen-1.5B como juez además del M1. Reutilizar M1 lo evita.

In [ ]:
# ============================================================
# Cómo elegir el juez (Dimensión 2) sin reventar la RAM de Colab
# ============================================================
# En free Colab (T4, ~12 GB de RAM) NO caben DOS modelos de 1.5B a la vez.
# Por eso, la opción RECOMENDADA es REUTILIZAR el M1 ya cargado como juez:
# no carga ningún modelo extra -> casi no consume RAM adicional.
#
# Opciones:
#   USAR_MISMO_MODELO_JUEZ = True  -> el M1 (extractor) también hace de juez. (RECOMENDADO)
#   USAR_MISMO_MODELO_JUEZ = False -> usa un juez Qwen separado o el minijuez.
#   JUEZ_LOCAL_APARTE     = True   -> junto con USAR_MISMO_MODELO_JUEZ=False, carga el
#                                     juez Qwen/Qwen2.5-0.5B-Instruct por separado.
# ============================================================
USAR_MISMO_MODELO_JUEZ = True   # RECOMENDADO para free Colab
JUEZ_LOCAL_APARTE = False       # True solo si quieres un juez separado (más RAM)

if USAR_MISMO_MODELO_JUEZ:
    # Reutiliza el mismo objeto `system` (M1) como juez: costo extra en RAM ≈ 0.
    judge_model = system
    judge_backend = {"kind": "local", "judge_model": judge_model}
elif JUEZ_LOCAL_APARTE:
    # Carga un Qwen pequeño (0.5B) aparte para juzgar.
    judge_model = make_judge_model()  # Qwen/Qwen2.5-0.5B-Instruct o JUDGE_MODEL
    judge_backend = {"kind": "local", "judge_model": judge_model}
else:
    # Minijuez determinista offline (sin modelo, solo para pruebas del flujo).
    judge_backend = {"kind": "mini"}

print("Backend de juez:", judge_backend["kind"])

### 8.3 Correr el harness

Ejecutamos las 3 dimensiones sobre el eval_set con el sistema cargado.

In [ ]:
scorecard = harness(EVAL_SET, system, judge_backend=judge_backend, umbral_judge=4)
imprimir_scorecard(scorecard)

### 8.4 Inspección de un caso concreto

Podemos ver a detalle cómo se calcula cada dimensión para un ejemplo puntual (p.ej. una
trampa out-of-domain) y qué oro el sistema debe cumplir.

In [ ]:
idx_trampa = next(i for i, e in enumerate(EVAL_SET) if e["expected_response"] == {})
ejemplo = EVAL_SET[idx_trampa]
print("Mensaje del cliente :", ejemplo["input"])
print("Gold (esperado)     :", ejemplo["expected_response"])
print("Criterio del campo  :", ejemplo["criterion"])
print("Salida del sistema  :", scorecard["d3_cumplimiento_dominio"]["detalles"][idx_trampa]["salida"])
print("Cumple D3 (vacío)   :", scorecard["d3_cumplimiento_dominio"]["detalles"][idx_trampa]["cumple"])

---

## 9. Lectura honesta del scorecard

- **D1 vs D2/D3:** D1 mide la coincidencia literal (exact-match / F1), que penaliza duro
  matices semánticos (p.ej. `oversized` vs `holgado`). El juez (D2) captura esos matices;
  por eso D2 suele puntuar más alto que el exact-match.
- **D3 es la métrica del campo:** la *fracción de cumplimiento* es el número que resume si
  el sistema cumple con los criterios que definimos como "buena respuesta" en el eval_set.
- **Trampas out-of-domain:** estas redundan en D1 y D3 (el JSON no puede coincidir con `{}`
  si se inventa algo), pero D3 añade la verificación específica que es independiente del juez.